
USTAR Moving Point Detection - Complete Workflow (Papale et al., 2006)

Comprehensive example demonstrating:
- USTAR threshold detection with forward/back modes
- Bootstrap uncertainty estimation
- Visualization of NEE response to USTAR stratification


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add parent package to path for direct execution
sys.path.insert(0, str(Path(__file__).parent.parent.parent))


def ustar_mp_complete_workflow():
    """
    Complete USTAR detection workflow with detection, uncertainty, and visualization.

    Steps:
    1. Load example data (LAE site)
    2. Detect USTAR thresholds & estimate uncertainty via bootstrap resampling
    3. Visualize NEE response to USTAR stratification by temperature class
    """
    import diive as dv

    print("\n" + "=" * 80)
    print("USTAR Moving Point Detection - Complete Workflow")
    print("Papale et al. (2006)")
    print("=" * 80)

    # =========================================================================
    # STEP 1: LOAD DATA
    # =========================================================================
    print("\n[STEP 1] Loading example data...")
    df = dv.load_exampledata_parquet_lae()

    # Use 2017-2018 for comprehensive analysis
    # locs = (df.index.year >= 2017) & (df.index.year <= 2018)
    # df = df.loc[locs].copy()

    print(f"  Loaded {len(df)} records ({df.index.year.min()}-{df.index.year.max()})")

    # Column names
    NEE_COL = "NEE_L3.1_L3.2_QCF_IRGA72"
    TA_COL = "TA_T1_47_1_gfXG_IRGA72"
    USTAR_COL = "USTAR_IRGA72"
    SW_IN = "SW_IN_T1_47_1_gfXG_IRGA72"

    # =========================================================================
    # STEP 2: DETECT USTAR THRESHOLDS & BOOTSTRAP UNCERTAINTY
    # =========================================================================
    print("\n[STEP 2] Detecting USTAR thresholds with bootstrap uncertainty (30 iterations)...")

    detector = dv.UstarMovingPointDetection(
        df,
        nee_col=NEE_COL,
        ta_col=TA_COL,
        ustar_col=USTAR_COL,
        swin_col=SW_IN,
        ta_classes_count=7,
        ustar_classes_count=20,
        bootstrapping_times=30,
        verbose=1
    )

    thresholds = detector.detect()
    print("\n" + detector.summary())

    # Get annual threshold (conservative approach: maximum across seasons)
    annual = detector.get_annual_thresholds()
    print("\n" + "=" * 75)
    print("Annual Thresholds (conservative approach: maximum of seasonal values)")
    print("=" * 75)
    print(f"Forward Mode: {annual['forward_mode']:.4f} m/s")
    print(f"Back Mode:    {annual['back_mode']:.4f} m/s")

    # Bootstrap uncertainty estimation
    stats = detector.bootstrap()

    print("\nBootstrap Statistics (mean ± std, [5%-95% CI]):")
    print("=" * 95)
    for idx, row in stats.iterrows():
        print(f"\n{idx}:")
        print(f"  Forward Mode: {row['forward_mean']:.4f} ± {row['forward_std']:.4f} "
              f"[{row['forward_p05']:.4f}, {row['forward_p95']:.4f}]")
        print(f"  Back Mode:    {row['back_mean']:.4f} ± {row['back_std']:.4f} "
              f"[{row['back_p05']:.4f}, {row['back_p95']:.4f}]")

    # =========================================================================
    # STEP 3: VISUALIZATION (Per-Temperature-Class Stratification)
    # =========================================================================
    print("\n[STEP 3] Visualizing NEE response to USTAR, stratified by temperature class...")

    season_groups = [[12, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
    season_names = ["Winter (DJF)", "Spring (MAM)", "Summer (JJA)", "Fall (SON)"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 13))
    axes = axes.flatten()

    df['month'] = df.index.month
    df['is_night'] = df[SW_IN] < 10  # Nighttime: SW_IN < 10 W/m² (pure respiration, no photosynthesis)

    for season_idx, (months, season_name) in enumerate(zip(season_groups, season_names)):
        ax = axes[season_idx]

        # Filter season data (nighttime only - respiration for USTAR threshold visualization)
        season_mask = (df['month'].isin(months)) & (df['is_night'])
        df_season = df[season_mask].dropna(subset=[NEE_COL, TA_COL, USTAR_COL, SW_IN])

        if len(df_season) < 100:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                   transform=ax.transAxes)
            ax.set_title(f"{season_name} - No data")
            continue

        # Sort by TA to create temperature classes (matching algorithm)
        df_sorted = df_season.sort_values(TA_COL).reset_index(drop=True)

        # Create TA classes (7 classes, matching algorithm)
        ta_classes_count = 7
        n_per_ta = len(df_sorted) // ta_classes_count

        colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, ta_classes_count))

        # Plot each temperature class separately
        for ta_class_idx in range(ta_classes_count):
            start_idx = ta_class_idx * n_per_ta
            if ta_class_idx == ta_classes_count - 1:
                end_idx = len(df_sorted)
            else:
                end_idx = (ta_class_idx + 1) * n_per_ta

            df_ta_class = df_sorted.iloc[start_idx:end_idx]

            if len(df_ta_class) < 50:
                continue

            # Create USTAR classes (15 quantile-based)
            n_ustar_classes = 15
            ustar_quantiles = np.linspace(0, 1, n_ustar_classes + 1)
            ustar_edges = df_ta_class[USTAR_COL].quantile(ustar_quantiles).values
            ustar_classes = np.digitize(df_ta_class[USTAR_COL].values, ustar_edges[1:-1])

            # Compute statistics per USTAR class
            ustar_means = []
            nee_means = []
            nee_stds = []
            ta_means = []

            for u_cls in range(1, n_ustar_classes + 1):
                mask = ustar_classes == u_cls
                if mask.sum() > 0:
                    ustar_means.append(df_ta_class.loc[mask, USTAR_COL].mean())
                    nee_means.append(df_ta_class.loc[mask, NEE_COL].mean())
                    nee_stds.append(df_ta_class.loc[mask, NEE_COL].std())
                    ta_means.append(df_ta_class.loc[mask, TA_COL].mean())

            if len(ustar_means) > 2:
                ustar_means = np.array(ustar_means)
                nee_means = np.array(nee_means)
                nee_stds = np.array(nee_stds)
                ta_means = np.array(ta_means)

                # Plot this TA class
                valid_idx = ~np.isnan(nee_means) & ~np.isnan(ustar_means)
                if sum(valid_idx) > 0:
                    ta_range = f"TA {ta_means.min():.1f}–{ta_means.max():.1f}°C"
                    ax.plot(ustar_means[valid_idx], nee_means[valid_idx], 'o-',
                           linewidth=2, markersize=6, label=ta_range,
                           color=colors[ta_class_idx], alpha=0.7)

        # Mark detected threshold
        threshold = thresholds.iloc[season_idx]['forward_mode']
        if not np.isnan(threshold) and threshold != 10.0:
            ax.axvline(threshold, color='red', linestyle='--', linewidth=2.5,
                      label=f'Forward Mode: {threshold:.3f} m/s')

        ax.set_xlabel('USTAR (m/s)', fontsize=11)
        ax.set_ylabel('Mean NEE (µmol/m²/s)', fontsize=11)
        ax.set_title(f'{season_name} - NEE vs USTAR (by Temperature Class)',
                    fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='best', ncol=1)

    plt.tight_layout()
    # Note: plt.show() displays the plot in interactive environments
    # In non-interactive mode (CI/testing), the plot is silently created
    try:
        plt.show()
    except (RuntimeError, ValueError):
        # Non-interactive backend, skip display
        pass
    plt.close('all')

    # =========================================================================
    # SUMMARY
    # =========================================================================
    print("\n" + "=" * 80)
    print("WORKFLOW COMPLETE")
    print("=" * 80)
    print(f"\nKey Results:")
    print(f"  Forward thresholds: {thresholds['forward_mode'].round(4).values} m/s")
    print(f"  Back thresholds:    {thresholds['back_mode'].round(4).values} m/s")
    print(f"\nRecommendation: Use forward mode thresholds for ecosystem flux filtering")
    print(f"(forward mode typically ~0.1-0.2 m/s for stable CO2 coupling)")

    return {
        'detector': detector,
        'thresholds': thresholds,
        'bootstrap_stats': stats
    }


if __name__ == '__main__':
    results = ustar_mp_complete_workflow()